# Baselines: DummyClassifier e Regressão Logística

Treinamento de modelos baseline para previsão de churn em telecomunicações.  


**Modelos:** DummyClassifier e Regressão Logística  
**Métricas monitoradas:** Accuracy, AUC-ROC, PR-AUC, F1, F1-weighted


In [1]:
%pip install mlflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# ===============================
# 1. Imports
# ===============================
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

import mlflow
import mlflow.sklearn


In [17]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    NOTEBOOKS_DIR = CURRENT_DIR
else:
    NOTEBOOKS_DIR = CURRENT_DIR / "notebooks"

MLFLOW_DB = NOTEBOOKS_DIR / "mlflow.db"
MLRUNS_DIR = NOTEBOOKS_DIR / "mlruns"

MLRUNS_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.as_posix()}")
mlflow.set_experiment("churn-model-comparison")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow artifacts dir:", MLRUNS_DIR)

MLflow tracking URI: sqlite:///c:/Users/fequi/OneDrive/Ambiente de Trabalho/Repos/tech-challenge/Tech-Challenge-Fase1/notebooks/mlflow.db
MLflow artifacts dir: c:\Users\fequi\OneDrive\Ambiente de Trabalho\Repos\tech-challenge\Tech-Challenge-Fase1\notebooks\mlruns


In [18]:
DATA_PATH = Path("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset carregado de: {DATA_PATH.resolve()}")

Dataset carregado de: C:\Users\fequi\OneDrive\Ambiente de Trabalho\Repos\tech-challenge\Tech-Challenge-Fase1\data\WA_Fn-UseC_-Telco-Customer-Churn.csv


In [19]:

# ===============================
# 3. Preparação dos dados
# ===============================
# Corrigir TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Preencher missing
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

# Criar target binária
df["target"] = df["Churn"].map({"Yes": 1, "No": 0})

# Separar X e y
X = df.drop(columns=["Churn", "target", "customerID"])
y = df["target"]

In [20]:
# ===============================
# 4. Split estratificado
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Distribuição treino:")
print(y_train.value_counts(normalize=True))


Distribuição treino:
target
0    0.734647
1    0.265353
Name: proportion, dtype: float64


In [21]:
# ===============================
# 5. Pipeline de pré-processamento
# ===============================
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])


In [22]:
# ===============================
# 6. Função de avaliação
# ===============================
def evaluate(y_true, y_pred, y_proba):
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred)
    }



In [23]:
# ===============================
# 7. Modelos baseline
# ===============================
models = {
    "dummy": DummyClassifier(strategy="most_frequent"),
    "logistic": LogisticRegression(max_iter=1000)
}


In [24]:
# ===============================
# 8. Treino + avaliação + MLflow
# ===============================
results = {}

for name, model in models.items():
    
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    with mlflow.start_run(run_name=name):
        
        pipeline.fit(X_train, y_train)
        
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        
        metrics = evaluate(y_test, y_pred, y_proba)
        results[name] = metrics
        
        # Log MLflow
        mlflow.log_param("model", name)
        
        for metric_name, value in metrics.items():
            mlflow.log_metric(metric_name, value)
        
        mlflow.sklearn.log_model(pipeline, "model")


c:\Users\fequi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
2026/04/28 22:02:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/28 22:02:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/28 22:02:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/28 22:02:59 WARNING mlflow.skle

In [25]:
# ===============================
# 9. Comparação final
# ===============================
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="roc_auc", ascending=False)

print("\n=== RESULTADOS BASELINE ===")
display(results_df)





=== RESULTADOS BASELINE ===


,roc_auc,pr_auc,f1,precision,recall
logistic,0.841874,0.633388,0.604046,0.657233,0.558824
dummy,0.500000,0.265436,0.000000,0.000000,0.000000


In [26]:
# ===============================
# 10. Interpretação rápida
# ===============================
print("\n=== INTERPRETAÇÃO ===")
print("Dummy representa baseline ingênuo (sem aprendizado).")
print("Logistic mostra se existe sinal real nos dados.")

if results_df.loc["logistic", "roc_auc"] > 0.7:
    print("\nExiste sinal preditivo relevante")
else:
    print("\nSinal fraco → revisar features / dados")


=== INTERPRETAÇÃO ===
Dummy representa baseline ingênuo (sem aprendizado).
Logistic mostra se existe sinal real nos dados.

Existe sinal preditivo relevante
